# 02 (Kaggle) - Train forward hi->en Transformer (RoPE)

Kaggle T4 driver. Trains the **RoPE** forward model on the corpus `01_data` built (uploaded as a Kaggle Dataset).

**Before running:**
- **Settings -> Accelerator -> GPU T4**, and **Settings -> Internet -> ON** (git clone + pip + Kaggle API need it).
- **Attach the corpus dataset** (the 7 files: `train.labse.*`, `dev.clean.*`, `test.clean.*`, `spm_unigram_16000.model`) and set `CORPUS_SLUG` below.
- For **autopush/resume**: add your Kaggle API token via **Add-ons -> Secrets** as `KAGGLE_USERNAME` and `KAGGLE_KEY`, set `CKPT_SLUG` below, and on the 2nd+ session also **attach the checkpoint dataset** so this notebook can resume from it.

Kaggle caps ~12 h/session and ~30 h/week; the T4 run is ~16-40 h, so expect 2-3 sessions. The startup cell copies prior checkpoints in (**resume**), the final cell pushes them back out (**autopush**). Interrupt training before the 12 h limit, then run the push cell to snapshot.

## 1. Repo + install

In [ ]:
import os, sys

REPO_URL = "https://github.com/Rishi-Jain-27/hindi-translator.git"
REPO_DIR = "/kaggle/working/hindi-translator"

# internet must be ON (Settings -> Internet)
!test -d {REPO_DIR} && git -C {REPO_DIR} pull || git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.abspath("src"))
print("cwd:", os.getcwd())

In [ ]:
!pip install -e .

In [ ]:
import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 2. Config + paths (fill in your dataset slugs)

In [ ]:
from nmt.config import DataConfig, ModelConfig, TrainConfig

CORPUS_SLUG = "rishijain27/hindi-translator-dataset"   # corpus dataset (attach it)
CKPT_SLUG   = "rishijain27/hindi-rope-ckpts"           # checkpoint dataset for autopush/resume (1st push creates it; rename if you like)

DATA_DIR = f"/kaggle/input/{CORPUS_SLUG.split('/')[-1]}"   # attached corpus (read-only)
CKPT_DIR = "/kaggle/working/ckpts"                        # writable; FLAT *.pt files (pushed to CKPT_SLUG)

data_cfg = DataConfig(cache_dir=DATA_DIR)                  # corpus files read from here
model_cfg = ModelConfig(pos_encoding="rope")              # base + RoPE; fresh run
train_cfg = TrainConfig(
    out_dir=CKPT_DIR,
    max_steps=100_000,        # upper bound; early stopping usually halts sooner
    warmup_steps=4000,
    eval_every=2000,
    ckpt_every=2000,          # rotation keeps last 2 + best.pt; autopush cadence is per-session
    log_every=50,
)
print("data:", DATA_DIR, "| ckpts:", CKPT_DIR, "| pos_encoding:", model_cfg.pos_encoding)

## 3. Resume - copy any prior checkpoints into the working dir

In [ ]:
import glob, shutil
os.makedirs(CKPT_DIR, exist_ok=True)

# if the checkpoint dataset is attached (2nd+ session), copy its *.pt into CKPT_DIR so the loop auto-resumes
PRIOR = f"/kaggle/input/{CKPT_SLUG.split('/')[-1]}"
if os.path.isdir(PRIOR):
    for f in glob.glob(os.path.join(PRIOR, "*.pt")):
        shutil.copy(f, CKPT_DIR)
    print("resumed from:", sorted(os.listdir(CKPT_DIR)))
else:
    print("no checkpoint dataset attached -> fresh run")

## 4. Tokenizer + data loaders

In [ ]:
from nmt.data.tokenizer import Tokenizer
from nmt.data.dataset import make_dataloader

tok = Tokenizer.load(os.path.join(DATA_DIR, f"spm_{data_cfg.tokenizer_model}_{data_cfg.vocab_size}.model"))
print("tokenizer vocab:", tok.vocab_size)

train_loader = make_dataloader(data_cfg, tok,
    os.path.join(DATA_DIR, "train.labse.hi"), os.path.join(DATA_DIR, "train.labse.en"),
    max_tokens=train_cfg.max_tokens, train=True)
dev_loader = make_dataloader(data_cfg, tok,
    os.path.join(DATA_DIR, "dev.clean.hi"), os.path.join(DATA_DIR, "dev.clean.en"),
    max_tokens=train_cfg.max_tokens, train=False)
print("train batches:", len(train_loader), "| dev batches:", len(dev_loader))

## 5. Build model

In [ ]:
from nmt.model.transformer import build_model
model = build_model(model_cfg, tok)
print("params:", f"{sum(p.numel() for p in model.parameters())/1e6:.1f}M", "| pos:", model_cfg.pos_encoding)

## 6. Train

Auto-resumes from any copied checkpoints. **Interrupt before the 12 h limit, then run the autopush cell below to snapshot.**

In [ ]:
from nmt.train.loop import Trainer
trainer = Trainer(train_cfg, model, train_loader, dev_loader, tok, out_dir=CKPT_DIR)
trainer.train()
print("done. step:", trainer.step, "| best dev nll:", trainer.best)

## 7. Autopush checkpoints to the Kaggle dataset

Run **after training (or after interrupting it)** to push `CKPT_DIR` to `CKPT_SLUG` so the next session can resume. First run **creates** the dataset; later runs push a new **version**. Needs `KAGGLE_USERNAME` / `KAGGLE_KEY` in Add-ons -> Secrets. Safe to run mid-run too (interrupt -> push -> resume next session).

In [ ]:
import json, subprocess
from kaggle_secrets import UserSecretsClient

# auth: pull the Kaggle API token from Secrets into the env vars the kaggle CLI reads
sec = UserSecretsClient()
os.environ["KAGGLE_USERNAME"] = sec.get_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = sec.get_secret("KAGGLE_KEY")

# dataset metadata must live in the pushed folder
owner, name = CKPT_SLUG.split("/")
with open(os.path.join(CKPT_DIR, "dataset-metadata.json"), "w") as f:
    json.dump({"title": name, "id": CKPT_SLUG, "licenses": [{"name": "CC0-1.0"}]}, f)

# try a new version; if the dataset doesn't exist yet, create it (--dir-mode skip: flat *.pt only)
msg = f"rope ckpt step {trainer.step}"
r = subprocess.run(["kaggle", "datasets", "version", "-p", CKPT_DIR, "-m", msg, "--dir-mode", "skip"])
if r.returncode != 0:
    print("version failed (dataset likely doesn't exist yet) -> creating")
    subprocess.run(["kaggle", "datasets", "create", "-p", CKPT_DIR, "--dir-mode", "skip"])
print("pushed checkpoints to", CKPT_SLUG)

## 8. Next

Next session: re-run top-to-bottom (the resume cell pulls the pushed checkpoints back in). When dev-nll plateaus / early-stops, evaluate with **`04_decode_eval_kaggle`** (attach the same corpus + checkpoint datasets).